# Previsões de cinco transformers — sem treino

Este notebook **não faz fine-tuning, não treina um classificador e não utiliza as labels humanas
para produzir previsões**. Todos os emails em `EMAIL_DIR` são processados. O Excel é opcional e só
é lido depois das previsões, para calcular métricas e mostrar erros caso a caso.

| Modelo | Método sem treino |
|---|---|
| [NorBERTo-large](https://huggingface.co/Itau-Unibanco/NorBERTo-large) | Semelhança de embeddings com descrições fixas |
| [XLM-RoBERTa-base](https://huggingface.co/FacebookAI/xlm-roberta-base) | Semelhança de embeddings com descrições fixas |
| [XLM-RoBERTa-large-XNLI](https://huggingface.co/joeddav/xlm-roberta-large-xnli) | Zero-shot com a cabeça NLI já treinada no checkpoint |
| [Albertina 900M PT-PT](https://huggingface.co/PORTULAN/albertina-900m-portuguese-ptpt-encoder) | Semelhança de embeddings com descrições fixas |
| [BERTimbau Base](https://huggingface.co/neuralmind/bert-base-portuguese-cased) | Semelhança de embeddings com descrições fixas |

Os quatro encoders base não têm uma cabeça treinada para estas três categorias nem são modelos
de chat. A semelhança com descrições permite gerar uma previsão sem lhes ensinar exemplos,
mas é uma **heurística exploratória**, especialmente fraca para uma categoria residual como SPAM.
O ranking compara estes **checkpoints e métodos sem treino**, não o potencial após fine-tuning.
Os três scores de cada email somam 1. Nos encoders base, um softmax normaliza as semelhanças cosseno; no XNLI, a normalização vem do método zero-shot. Continuam a não ser probabilidades calibradas nem scores diretamente comparáveis entre métodos.

## Executar

1. Seleciona o kernel `.GB` no Jupyter/VS Code e reinicia-o se tinhas executado a versão anterior.
2. Confirma `EMAIL_DIR` e executa todas as células por ordem.
3. A primeira execução descarrega os modelos (vários GB). A inferência corre localmente, um modelo de cada vez.
4. Consulta a tabela por UID e os CSV em `research/benchmark_results/<data-hora>/`.

CPU é a opção inicial para compatibilidade. Podes escolher `mps` no Mac ou `cuda` numa máquina NVIDIA.
Albertina 900M requer mais memória. Falhas de carregamento são registadas, sem substituir o checkpoint.
Ter apenas uma label de encomenda **não bloqueia nenhuma previsão**.


In [ ]:
# Apenas se faltarem dependências; reinicia o kernel após instalar.
# %pip install "transformers>=4.48,<6" "torch>=2.6" sentencepiece protobuf numpy pandas openpyxl scikit-learn matplotlib ipykernel


In [ ]:
import gc
import hashlib
import html
import importlib.metadata
import json
import platform
import re
import time
import unicodedata
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer, set_seed

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'tests' / 'extraction.py').exists():
            return candidate
    raise FileNotFoundError('Abre o notebook dentro do projeto GlobalBrico.')

ROOT = find_root()
GROUND_TRUTH = ROOT / 'research/ground_truth/ground_truth_emails.xlsx'
SHEET = 'Revisão'
EMAIL_DIR = ROOT / 'research/extracted_emails'
RESULTS_ROOT = ROOT / 'research/benchmark_results'
CACHE_DIR = RESULTS_ROOT / 'embedding_cache'
LABELS = ['Pedido de Informação', 'Pedido de Encomenda', 'SPAM']
SEED = 42
MAX_LENGTH = 512           # mesmo orçamento em tokens; os tokenizers diferem
BATCH_SIZE = 1             # aumentar se houver memória disponível
DEVICE = 'cpu'             # 'cpu', 'mps' (Apple Silicon), 'cuda' (NVIDIA)
REMOVE_SPAM_PREFIX = True  # retira apenas marcações SPAM no início do assunto
USE_EMBEDDING_CACHE = True

MODELS = [
    {'name': 'NorBERTo-large', 'repo': 'Itau-Unibanco/NorBERTo-large', 'revision': 'main'},
    {'name': 'XLM-RoBERTa-base', 'repo': 'FacebookAI/xlm-roberta-base', 'revision': 'main'},
    {'name': 'XLM-RoBERTa-large-XNLI', 'repo': 'joeddav/xlm-roberta-large-xnli', 'revision': 'main'},
    {'name': 'Albertina 900M PT-PT', 'repo': 'PORTULAN/albertina-900m-portuguese-ptpt-encoder', 'revision': 'main'},
    {'name': 'BERTimbau Base', 'repo': 'neuralmind/bert-base-portuguese-cased', 'revision': 'main'},
]
# Para repetir exatamente uma execução, substitui 'main' pelos SHAs do manifesto exportado.
set_seed(SEED)
assert DEVICE in {'cpu', 'mps', 'cuda'}
if DEVICE == 'mps' and not torch.backends.mps.is_available():
    raise RuntimeError('MPS indisponível neste kernel; usa DEVICE="cpu".')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA indisponível neste kernel; usa DEVICE="cpu".')
if MAX_LENGTH > 512 or MAX_LENGTH < 32 or BATCH_SIZE < 1:
    raise ValueError('Para esta comparação usa 32 <= MAX_LENGTH <= 512 e BATCH_SIZE >= 1.')
PACKAGES = ['torch', 'transformers', 'tokenizers', 'sentencepiece', 'protobuf', 'numpy', 'pandas', 'scikit-learn', 'openpyxl', 'matplotlib']
VERSIONS = {p: importlib.metadata.version(p) for p in PACKAGES}
print('Python:', platform.python_version(), '| Dispositivo:', DEVICE)
display(pd.DataFrame(MODELS))

# Estas descrições são fixas e não contêm exemplos/labels do Excel.
# SPAM significa toda a correspondência fora dos dois tipos de pedidos.
HYPOTHESES = [
    'Este email solicita informações, preços, um orçamento ou esclarecimentos sobre produtos ou serviços.',
    'Este email faz ou confirma uma encomenda de produtos ou serviços.',
    'Este email não solicita informações nem faz uma encomenda; é outra correspondência ou spam.',
]


In [ ]:
def normalize(value):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value))).strip()

def normalize_uid(value):
    value = normalize(value)
    return re.sub(r'^(\d+)\.0$', r'\1', value)

def clean_subject(subject):
    subject = str(subject or '')
    if REMOVE_SPAM_PREFIX:
        subject = re.sub(r'^\s*(?:\[\s*spam\s*\]|\*+spam\*+|spam\b)[\s:_-]*', '', subject, flags=re.I)
    return subject.strip()

def body_text(email):
    if str(email.get('text') or '').strip():
        return email['text'].strip()
    raw = str(email.get('html') or '')
    raw = re.sub(r'<(script|style)\b[^>]*>.*?</\1>', ' ', raw, flags=re.I | re.S)
    return html.unescape(re.sub(r'<[^>]+>', ' ', raw)).strip()

def load_emails(email_dir):
    records = []
    for source in sorted(email_dir.glob('*.json')):
        if source.name == 'summary.json':
            continue
        email = json.loads(source.read_text(encoding='utf-8'))
        uid = normalize_uid(email.get('uid', ''))
        if not uid or uid == 'None':
            raise ValueError(f'JSON sem UID: {source}')
        subject = clean_subject(email.get('subject'))
        body = body_text(email)
        if not (subject or body):
            raise ValueError(f'UID {uid}: sem assunto nem corpo para classificação.')
        text = '\n'.join([f'Assunto: {subject}', f'Remetente: {email.get("from", "")}', body])
        records.append({'uid': uid, 'subject': subject, 'text': text,
                        'source': str(source.relative_to(ROOT))})
    if not records:
        raise ValueError(f'Não encontrei emails em {email_dir}.')
    frame = pd.DataFrame(records)
    if frame['uid'].duplicated().any():
        raise ValueError('UIDs repetidos: separa as pastas ou usa um identificador pasta+UID.')
    return frame

df = load_emails(EMAIL_DIR)
print(f'{len(df)} emails para prever. O Excel ainda não foi lido.')
display(df[['uid', 'subject']])


## Inferência

Para os encoders base, calcula-se a média dos embeddings dos tokens do email e de cada descrição,
excluindo padding e tokens especiais. Um softmax transforma as três semelhanças em scores que somam 1; a categoria com maior score é escolhida.
Nenhum parâmetro é ajustado. Para XNLI, escolhe-se a hipótese com maior score de entailment.

O assunto, remetente e corpo são os únicos inputs. Por defeito remove-se o prefixo SPAM do assunto
para não copiar a marcação de um filtro anterior. O conteúdo dos anexos não está nos JSON.
Todos usam até `MAX_LENGTH` tokens; a hipótese NLI também ocupa parte desse limite.
Registam-se os emails truncados, a revisão exata do modelo e os tempos desta execução.


In [ ]:
def release_memory():
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    if DEVICE == 'mps': torch.mps.empty_cache()

def synchronize():
    if DEVICE == 'cuda': torch.cuda.synchronize()
    if DEVICE == 'mps': torch.mps.synchronize()

def resolve_model(spec):
    config = AutoConfig.from_pretrained(spec['repo'], revision=spec['revision'], trust_remote_code=False)
    sha = getattr(config, '_commit_hash', None)
    if not sha:
        raise RuntimeError('Não foi possível obter o SHA do modelo para registar a revisão.')
    return config, sha

def load_encoder_model(spec, config, sha):
    kwargs = {'config': config, 'revision': sha, 'trust_remote_code': False}
    if config.model_type == 'modernbert':
        config.reference_compile = False
        kwargs['attn_implementation'] = 'eager'
    return AutoModel.from_pretrained(spec['repo'], **kwargs).float().to(DEVICE).eval()

def extract_embeddings(spec, frame):
    config, sha = resolve_model(spec)
    identity = {'repo': spec['repo'], 'sha': sha, 'texts': frame['text'].tolist(),
                'uids': frame['uid'].tolist(), 'max_length': MAX_LENGTH,
                'pooling': 'mean_without_special_tokens_v1', 'dtype': 'float32',
                'device': DEVICE, 'batch_size': BATCH_SIZE, 'versions': VERSIONS}
    key = hashlib.sha256(json.dumps(identity, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = CACHE_DIR / f'{key}.npz'
    if USE_EMBEDDING_CACHE and cache.exists():
        with np.load(cache, allow_pickle=False) as saved:
            features = saved['features'].copy()
            lengths = saved['lengths'].copy()
            parameters = int(saved['parameters'].item())
        assert features.shape[0] == len(frame) and np.isfinite(features).all()
        assert lengths.shape == (len(frame),)
        return features, {'revision': sha, 'cache_hit': True, 'parameters': parameters,
                          'load_seconds': None, 'encode_seconds': None, 'seconds_per_email': None,
                          'truncated_emails': int((lengths > MAX_LENGTH).sum())}
    model = tokenizer = None
    try:
        start = time.perf_counter()
        tokenizer = AutoTokenizer.from_pretrained(spec['repo'], revision=sha, trust_remote_code=False)
        model = load_encoder_model(spec, config, sha)
        parameters = sum(p.numel() for p in model.parameters())
        synchronize()
        load_seconds = time.perf_counter() - start
        lengths = np.array([len(tokenizer(t, truncation=False, verbose=False)['input_ids']) for t in frame['text']])
        parts = []
        synchronize()
        start = time.perf_counter()
        with torch.inference_mode():
            for offset in range(0, len(frame), BATCH_SIZE):
                batch = tokenizer(frame['text'].iloc[offset:offset+BATCH_SIZE].tolist(), padding=True,
                                  truncation=True, max_length=MAX_LENGTH, return_tensors='pt', return_special_tokens_mask=True)
                special = batch.pop('special_tokens_mask').to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                hidden = model(**batch).last_hidden_state
                mask = (batch['attention_mask'] * (1 - special)).unsqueeze(-1).to(hidden.dtype)
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
                parts.append(pooled.cpu().numpy())
        synchronize()
        encode_seconds = time.perf_counter() - start
        features = np.concatenate(parts).astype(np.float32)
        assert features.shape[0] == len(frame) and np.isfinite(features).all()
        if USE_EMBEDDING_CACHE:
            np.savez_compressed(cache, features=features, lengths=lengths, parameters=parameters)
        return features, {'revision': sha, 'cache_hit': False, 'parameters': parameters,
                          'load_seconds': load_seconds, 'encode_seconds': encode_seconds,
                          'seconds_per_email': encode_seconds / len(frame),
                          'truncated_emails': int((lengths > MAX_LENGTH).sum())}
    finally:
        del model, tokenizer
        release_memory()

def make_predictions(frame, name, method, scores):
    scores = np.asarray(scores, dtype=np.float64)
    if scores.shape != (len(frame), len(LABELS)) or not np.isfinite(scores).all():
        raise ValueError('Scores incompletos ou inválidos.')
    if method == 'cosine_descriptions':
        # Softmax preserva a classe com maior semelhança e normaliza as três opções.
        shifted = scores - scores.max(axis=1, keepdims=True)
        scores = np.exp(shifted)
    elif method != 'zero_shot_nli' or np.any(scores < 0):
        raise ValueError(f'Método ou scores inválidos: {method}')
    scores = scores / scores.sum(axis=1, keepdims=True)
    if not np.allclose(scores.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError('Os scores das três categorias não somam 1.')
    result = frame[['uid', 'subject', 'source']].copy()
    result['model'] = name
    result['method'] = method
    result['predicted'] = np.asarray(LABELS)[scores.argmax(axis=1)]
    result['score'] = scores.max(axis=1)
    for i, label in enumerate(LABELS):
        result[f'score_{label}'] = scores[:, i]
    return result

def predict_similarity(frame, spec):
    descriptions = pd.DataFrame({'uid': [f'__description_{i}' for i in range(len(LABELS))], 'text': HYPOTHESES})
    inputs = pd.concat([frame[['uid', 'text']], descriptions], ignore_index=True)
    embeddings, info = extract_embeddings(spec, inputs)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms < 1e-12):
        raise ValueError('Embedding de norma zero; não é possível calcular semelhança.')
    unit = embeddings / norms
    scores = unit[:len(frame)] @ unit[len(frame):].T
    # O tempo de embeddings inclui as três descrições, além dos emails.
    if info['encode_seconds'] is not None:
        info['seconds_per_email'] = info['encode_seconds'] / len(frame)
    return make_predictions(frame, spec['name'], 'cosine_descriptions', scores), info


In [ ]:
def predict_nli(frame, spec):
    config, sha = resolve_model(spec)
    entailment = [int(i) for i, label in config.id2label.items() if str(label).lower().startswith('entail')]
    if len(entailment) != 1:
        raise ValueError(f'Configuração NLI sem label entailment inequívoca: {config.id2label}')
    tokenizer = model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(spec['repo'], revision=sha, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(spec['repo'], config=config, revision=sha,
                    trust_remote_code=False).float().to(DEVICE).eval()
        all_scores = []
        truncated = 0
        synchronize()
        start = time.perf_counter()
        with torch.inference_mode():
            for text in frame['text']:
                logits = []
                pair_lengths = [len(tokenizer(text, h, truncation=False, verbose=False)['input_ids']) for h in HYPOTHESES]
                truncated += int(max(pair_lengths) > MAX_LENGTH)
                for offset in range(0, len(HYPOTHESES), BATCH_SIZE):
                    hypotheses = HYPOTHESES[offset:offset + BATCH_SIZE]
                    batch = tokenizer([text] * len(hypotheses), hypotheses, padding=True,
                                      truncation='only_first', max_length=MAX_LENGTH, return_tensors='pt')
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    logits.extend(model(**batch).logits[:, entailment[0]].float().cpu().tolist())
                all_scores.append(torch.softmax(torch.tensor(logits), dim=0).numpy())
        synchronize()
        elapsed = time.perf_counter() - start
        probabilities = np.asarray(all_scores)
        predicted = np.asarray(LABELS)[probabilities.argmax(axis=1)]
        return make_predictions(frame, spec['name'], 'zero_shot_nli', probabilities), {
            'revision': sha, 'encode_seconds': elapsed, 'seconds_per_email': elapsed / len(frame),
            'truncated_emails': truncated}
    finally:
        del model, tokenizer
        release_memory()


In [ ]:
# Reexecutar esta célula inicia um novo ensaio, sem acumular previsões anteriores.
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = RESULTS_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
all_predictions, execution_log = [], []
for spec in MODELS:
    print(f"A prever com {spec['name']}...", flush=True)
    start = time.perf_counter()
    try:
        predictor = predict_nli if spec['repo'] == 'joeddav/xlm-roberta-large-xnli' else predict_similarity
        detail, info = predictor(df, spec)
        all_predictions.append(detail)
        execution_log.append({**spec, **info, 'method': detail['method'].iloc[0],
                              'status': 'ok', 'total_seconds': time.perf_counter() - start})
        slug = re.sub(r'[^a-zA-Z0-9]+', '_', spec['name'])
        detail.to_csv(RUN_DIR / f'predictions_{slug}.csv', index=False, encoding='utf-8-sig')
        print(f'  {len(detail)} previsões guardadas.')
    except Exception as error:
        execution_log.append({**spec, 'status': 'error', 'error': f'{type(error).__name__}: {error}'})
        print(f'  FALHOU: {type(error).__name__}: {error}')
    finally:
        release_memory()
        (RUN_DIR / 'execution_log.json').write_text(json.dumps(execution_log, ensure_ascii=False, indent=2), encoding='utf-8')

display(pd.DataFrame(execution_log))
if not all_predictions:
    raise RuntimeError(f'Nenhum modelo terminou. Consulta {RUN_DIR / "execution_log.json"}.')
predictions = pd.concat(all_predictions, ignore_index=True)
comparison = predictions.pivot(index='uid', columns='model', values='predicted')
# Uma coluna explícita para cada modelo, incluindo eventuais falhas.
comparison = comparison.reindex(columns=[m['name'] for m in MODELS]).fillna('SEM PREVISÃO — consultar log')
comparison = df.set_index('uid')[['subject']].join(comparison)
predictions.to_csv(RUN_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')
comparison.to_csv(RUN_DIR / 'comparison_by_uid.csv', encoding='utf-8-sig')
print(f'Modelos concluídos: {len(all_predictions)}/{len(MODELS)}')
display(comparison)


## Avaliação opcional — só depois das previsões

Apenas a coluna **Label correta** do Excel é usada. As labels previstas anteriormente são ignoradas.
Uma só encomenda permite medir se esse caso foi acertado, mas não estimar recall de encomendas com confiança.
Se uma classe não tiver nenhum exemplo, o seu recall e F1 ficam indisponíveis e não há macro-F1 de três classes.
Sem Excel, ou com todas as labels em branco, as previsões continuam disponíveis.


In [ ]:
def read_labels(excel_path):
    raw = pd.read_excel(excel_path, sheet_name=SHEET, header=None, dtype=str, keep_default_na=False)
    header = next((i for i in range(min(20, len(raw)))
                   if {'UID', 'Label correta'}.issubset({normalize(v) for v in raw.iloc[i]})), None)
    if header is None:
        raise ValueError('Colunas UID e Label correta não encontradas.')
    table = raw.iloc[header + 1:].copy()
    table.columns = [normalize(v) for v in raw.iloc[header]]
    labels = table[['UID', 'Label correta']].rename(columns={'UID': 'uid', 'Label correta': 'label'})
    labels['uid'] = labels['uid'].map(normalize_uid)
    labels['label'] = labels['label'].map(normalize)
    labels = labels.loc[(labels['uid'] != '') & (labels['label'] != '')].copy()
    mapping = {label.casefold(): label for label in LABELS}
    invalid = labels.loc[~labels['label'].str.casefold().isin(mapping), 'label'].unique()
    if len(invalid):
        raise ValueError(f'Labels desconhecidas: {invalid.tolist()}')
    if labels['uid'].duplicated().any():
        raise ValueError('UIDs repetidos no Excel.')
    labels['label'] = labels['label'].str.casefold().map(mapping)
    return labels

evaluation_status = 'not_run'
metric_rows, report_rows = [], []
try:
    truth = read_labels(GROUND_TRUTH)
    evaluated = predictions.merge(truth, on='uid', how='inner', validate='many_to_one')
    if evaluated.empty:
        print('Não existem labels humanas correspondentes. Previsões guardadas na mesma.')
        evaluation_status = 'no_matching_labels'
    else:
        evaluation_status = 'ok'
        evaluated['correct'] = evaluated['label'] == evaluated['predicted']
        print(f"Avaliação em {evaluated['uid'].nunique()}/{len(df)} emails com label humana.")
        display(truth.loc[truth['uid'].isin(df['uid'])].groupby('label').size().reindex(LABELS, fill_value=0))
        for name, group in evaluated.groupby('model', sort=False):
            y_true, y_pred = group['label'].to_numpy(), group['predicted'].to_numpy()
            complete = set(y_true) == set(LABELS)
            requests = y_true != 'SPAM'
            spam = y_true == 'SPAM'
            metric_rows.append({'model': name, 'method': group['method'].iloc[0], 'n_evaluated': len(group),
                'macro_f1_3_classes': f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0) if complete else np.nan,
                'accuracy': accuracy_score(y_true, y_pred),
                'requests_as_spam': int(np.sum(requests & (y_pred == 'SPAM'))),
                'spam_as_request': int(np.sum(spam & (y_pred != 'SPAM')))})
            report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
            for label in LABELS:
                values = dict(report[label])
                if values['support'] == 0:
                    values['recall'] = values['f1-score'] = np.nan
                report_rows.append({'model': name, 'label': label, **values})
            matrix = confusion_matrix(y_true, y_pred, labels=LABELS)
            fig, ax = plt.subplots(figsize=(6, 5))
            ConfusionMatrixDisplay(matrix, display_labels=['Informação', 'Encomenda', 'SPAM']).plot(ax=ax, colorbar=False, cmap='Blues')
            ax.set(title=name, xlabel='Prevista', ylabel='Correta')
            fig.tight_layout()
            fig.savefig(RUN_DIR / f"confusion_{re.sub(r'[^a-zA-Z0-9]+', '_', name)}.png", dpi=160)
            plt.show()
            plt.close(fig)
        metrics = pd.DataFrame(metric_rows).sort_values('macro_f1_3_classes', ascending=False, na_position='last')
        display(metrics)
        display(pd.DataFrame(report_rows))
        errors = evaluated.loc[~evaluated['correct']]
        display(errors[['uid', 'model', 'label', 'predicted', 'subject']])
        metrics.to_csv(RUN_DIR / 'metrics.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame(report_rows).to_csv(RUN_DIR / 'per_class.csv', index=False, encoding='utf-8-sig')
        errors.to_csv(RUN_DIR / 'errors.csv', index=False, encoding='utf-8-sig')
        evaluated.to_csv(RUN_DIR / 'evaluated_predictions.csv', index=False, encoding='utf-8-sig')
except (FileNotFoundError, ValueError, OSError) as error:
    evaluation_status = f'skipped: {type(error).__name__}: {error}'
    print('Avaliação não realizada:', error)
    print('As previsões dos modelos já estão guardadas e não dependem do Excel.')


In [ ]:
manifest = {
    'run_id': run_id, 'training': False, 'n_emails': len(df), 'models': MODELS,
    'labels': LABELS, 'descriptions': HYPOTHESES, 'device': DEVICE, 'packages': VERSIONS,
    'seed': SEED, 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
    'remove_spam_prefix': REMOVE_SPAM_PREFIX,
    'dataset_sha256': hashlib.sha256(json.dumps(df[['uid', 'text']].to_dict('records'), ensure_ascii=False, sort_keys=True).encode()).hexdigest(),
    'ground_truth': str(GROUND_TRUTH), 'evaluation_status': evaluation_status,
    'complete_comparison': len(all_predictions) == len(MODELS), 'execution_log': execution_log,
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Resultados:', RUN_DIR)
print('Previsões por email: comparison_by_uid.csv')
if len(all_predictions) < len(MODELS):
    print('Comparação incompleta: um ou mais modelos falharam. Consulta execution_log.json.')


## Ler os resultados

`comparison_by_uid.csv` mostra uma coluna por transformer. `predictions.csv` contém o método,
a categoria escolhida e os três scores normalizados de cada previsão (soma = 1). `errors.csv`, quando existe ground truth,
permite rever os erros. Não há divisões treino/teste, porque nenhuma label é usada na inferência.

Compara macro-F1, recall por classe e, em especial, pedidos incorretamente classificados como SPAM.
Mantém as descrições fixas durante este ensaio: ajustá-las depois de ver os erros transforma esta
amostra em dados de desenvolvimento e requer novos emails para uma avaliação independente.

As condições de utilização também contam numa seleção futura: a model card consultada de
NorBERTo-large declara CC-BY-NC-SA-4.0. O resultado deste ensaio não estima desempenho após fine-tuning.
